In [ ]:
from config import *
import requests

In [ ]:
def fetch_patents(ipc_list, limit_per_class=1000):
    url = "https://searchplatform.rospatent.gov.ru/patsearch/v0.2/search"
    headers = Config.EXAMPLE_HEADERS

    all_hits = []

    for ipc in ipc_list:
        payload = {
            "q": "*",
            "filter": {
                "classification.ipc_subclass": {"values": [ipc]}
            },
            "limit": limit_per_class
        }
        r = requests.post(url, headers=headers, json=payload)
        hits = r.json().get("hits", [])

        hits = [h for h in hits if h.get(
            "snippet", {}).get("description", "").strip()]
        all_hits.extend(hits)

    return all_hits


IPC_CLASSES = ["H04M", "G06F", "A61B", "F02K", "B60R"]

patents = fetch_patents(IPC_CLASSES)

In [ ]:
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader
from collections import defaultdict
import random


def build_sbert_train_data(patents, max_pairs_per_ipc=100):
    ipc_map = defaultdict(list)

    for p in patents:
        desc = p.get("snippet", {}).get("description", "").strip()
        ipc = p.get("snippet", {}).get(
            "classification", {}).get("ipc", "").strip()
        if desc and ipc:
            ipc_map[ipc].append(desc)

    positives = []
    negatives = []
    ipc_list = list(ipc_map.keys())

    for ipc, descs in ipc_map.items():
        for i in range(min(len(descs), max_pairs_per_ipc)):
            for j in range(i + 1, min(len(descs), max_pairs_per_ipc)):
                positives.append(InputExample(
                    texts=[descs[i], descs[j]], label=1.0))

    for _ in range(len(positives)):
        ipc1, ipc2 = random.sample(ipc_list, 2)
        d1 = random.choice(ipc_map[ipc1])
        d2 = random.choice(ipc_map[ipc2])
        negatives.append(InputExample(texts=[d1, d2], label=0.0))

    return positives + negatives


train_samples = build_sbert_train_data(patents)
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)

model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
train_loss = losses.CosineSimilarityLoss(model)

model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=1, warmup_steps=100)

model.save("fine_tuned_sbert_patents")

In [25]:
from sentence_transformers import SentenceTransformer
from torch.nn.functional import cosine_similarity
import torch
import random
from tqdm import tqdm

random.seed(42)

queries = [
    ("Телефон с термометром", "Смартфон с функцией измерения температуры тела"),
    ("Устройство для очистки воздуха",
     "Фильтр с HEPA и угольным слоем для жилых помещений"),
    ("Ракета с выдвижными стабилизаторами",
     "Ракетный двигатель с раскрываемыми стабилизаторами"),
    ("Аварийная связь с солнечным питанием",
     "Передатчик с солнечной зарядкой и функцией SOS"),
    ("Голосовое управление в авто",
     "Ассистент для навигации с распознаванием речи в машине"),
    ("Автоматический дозатор антисептика",
     "ИК-дозатор с резервуаром для дезинфекции рук"),
    ("Очки с дополненной реальностью для инженеров",
     "AR-устройство с визуализацией техдокументов"),
    ("Трекинг сна и пульса в часах", "Наручник с ЧСС и фазами сна"),
    ("Фильтр воды с самоочисткой", "Мембранный фильтр с регенерацией"),
    ("Охлаждение процессора", "Тепловые трубки для отвода тепла от чипа"),
    ("Оптимизация маршрутов доставки", "ПО для логистики с прогнозом трафика"),
    ("Пьезоэлектрический генератор", "Генератор энергии от давления на пьезокристалл"),
    ("Сенсор падения в телефоне", "Акселерометр для фиксации удара"),
    ("Разблокировка по отпечатку", "Сканер пальца в мобильном устройстве"),
    ("Система полива по влажности", "Контроллер подачи воды на основе датчиков почвы"),
    ("Голос в текст", "Программа для расшифровки речи в реальном времени"),
    ("Панель управления домом", "Сенсорный интерфейс для управления IoT-устройствами"),
    ("Камера ночного видения в смартфоне",
     "ИК-камера с обработкой низкой освещенности"),
    ("Шумоподавление в микрофоне", "Микрофон с подавлением фоновых шумов"),
    ("Контроллер солнечной панели", "Модуль управления зарядом от солнечного элемента")
]

base_hard = [
    "Смартфон с камерой и GPS",
    "Очиститель воздуха с ароматизатором",
    "Ракета с фиксированными стабилизаторами",
    "Радиоприемник на батарейках",
    "Навигация с тачскрина без голосового ввода",
    "Ручной дозатор с кнопкой",
    "Очки виртуальной реальности для игр",
    "Фитнес-браслет с шагомером",
    "Фильтр воды с ультрафиолетом",
    "Вентилятор с медной решёткой",
    "Программа учёта запасов на складе",
    "Генератор на базе индукционного тока",
    "Модуль GPS для телефона",
    "Разблокировка по паролю",
    "Таймер полива без датчиков",
    "Редактор текста",
    "Пульт управления для кондиционера",
    "Камера со вспышкой",
    "Простой микрофон для подкастов",
    "Солнечный фонарь с аккумулятором"
]

hard_negatives = base_hard * 5

base_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
fine_model = SentenceTransformer("./fine_tuned_sbert_patents")


def evaluate_difficult_top1(model, name="Model"):
    correct_top1 = 0
    ranks = []

    print(name)
    for query, correct in tqdm(queries):
        candidates = hard_negatives.copy()
        candidates.append(correct)
        random.shuffle(candidates)

        emb_query = model.encode(query, convert_to_tensor=True)
        emb_cands = model.encode(candidates, convert_to_tensor=True)

        sims = cosine_similarity(emb_query, emb_cands)
        top_idx = torch.argmax(sims).item()
        correct_idx = candidates.index(correct)

        sorted_indices = torch.argsort(sims, descending=True).tolist()
        rank = sorted_indices.index(correct_idx) + 1
        ranks.append(rank)

        if top_idx == correct_idx:
            correct_top1 += 1

    recall = correct_top1 / len(queries)
    mean_rank = sum(ranks) / len(ranks)

    print(f"  Recall@1:  {recall:.2f}")
    print(f"  Mean Rank: {mean_rank:.2f}")


evaluate_difficult_top1(base_model, "Original SBERT")
evaluate_difficult_top1(fine_model, "Fine-tuned SBERT (IPC)")

Original SBERT


100%|██████████| 20/20 [00:03<00:00,  6.46it/s]


  Recall@1:  0.40
  Mean Rank: 11.00
Fine-tuned SBERT (IPC)


100%|██████████| 20/20 [00:03<00:00,  6.56it/s]

  Recall@1:  0.50
  Mean Rank: 9.75
